In [3]:
%cd /kaggle/input/beit3-main/beit3
%ls

/kaggle/input/beit3-main/beit3
datasets.py               modeling_finetune.py  README.md
engine_for_finetuning.py  modeling_utils.py     requirements.txt
get_started/              optim_factory.py      run_beit3_finetuning.py
glossary.py               randaug.py            utils.py


In [2]:
%%capture
!pip install -r requirements.txt

In [3]:
model_ckpt_path = '/kaggle/input/beit3/pytorch/beit3-vqa/1/beit3_base_indomain_patch16_480_vqa.pth'

train_csv_path = '/kaggle/input/foodvqa/train.csv'
val_csv_path = '/kaggle/input/foodvqa/validation.csv'
test_csv_path = '/kaggle/input/foodvqa/test.csv'

assets_path = '/kaggle/input/foodvqa/assets'

output_path = '/kaggle/working/'

mapping_path = '/kaggle/input/vivqa-dataset/vie2eng.json'

In [4]:
import math
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import json

from timm.models.layers import trunc_normal_ as __call_trunc_normal_
from timm.models.registry import register_model
import torch.nn.functional as F

from torchscale.architecture.config import EncoderConfig
# import utils

from torchvision import transforms
from PIL import Image

## Data preprocessing

In [5]:
train_df = pd.read_csv(train_csv_path)
val_df = pd.read_csv(val_csv_path)
test_df = pd.read_csv(test_csv_path)

In [6]:
train_df

,Image,Question,Answer
0,2e789c07ec72245,What color is the ground beef inside the enchi...,reddish-brown
1,2e789c07ec72245,What type of dish is shown in the image,enchiladas
2,2e789c07ec72245,What color is the rice next to the enchiladas,orange
3,2e789c07ec72245,Where are the sliced black olives placed,on top
4,2e789c07ec72245,What is the enchilada sauce and melted cheese ...,plate
...,...,...,...
34709,69fc8de514a3aba,What color is the bottom layer of the dessert,yellow
34710,69fc8de514a3aba,What type of dish is Zuppa Inglese,dessert
34711,69fc8de514a3aba,What is the state of the sponge cake in the de...,soaked
34712,69fc8de514a3aba,What color is the chocolate pudding layer,brown


In [9]:
def build_transform(input_size):
    transform = transforms.Compose([
        transforms.Resize((input_size, input_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    return transform

def to_image_tokens(image_path, input_size=384):
    transform = build_transform(input_size)
    image = Image.open(image_path).convert('RGB')
    image = transform(image)
    return image.unsqueeze(0)  # Add batch dimension

def to_text_tokens(text, tokenizer, max_len=64):
    tokens_orig = tokenizer.tokenize(text)
    token_ids = tokenizer.convert_tokens_to_ids(tokens_orig)
    
    if len(token_ids) > max_len - 2:
        token_ids = token_ids[:max_len - 2]
    
    tokens = [tokenizer.bos_token_id] + token_ids + [tokenizer.eos_token_id]
    num_tokens = len(tokens)
    padding_mask = [0] * num_tokens + [1] * (max_len - num_tokens)
    tokens_padded = tokens + [tokenizer.pad_token_id] * (max_len - num_tokens)

    padding_mask_tensor = torch.tensor(padding_mask).reshape(1, -1).to(device)
    token_ids_tensor = torch.tensor(tokens_padded).reshape(1, -1).to(device)

    return token_ids_tensor, padding_mask_tensor

## Model utils

In [10]:
from torchscale.architecture.encoder import Encoder
from torchscale.component.embedding import (
    PositionalEmbedding,
    TextEmbedding,
    VisionEmbedding,
)
from torchscale.component.multiway_network import MutliwayEmbedding


class BEiT3(nn.Module):
    def __init__(self, args, **kwargs):
        super().__init__()
        self.args = args
        assert args.multiway
        assert args.vocab_size > 0
        assert not args.share_encoder_input_output_embed
        self.text_embed = TextEmbedding(args.vocab_size, args.encoder_embed_dim)
        self.vision_embed = VisionEmbedding(
            args.img_size,
            args.patch_size,
            args.in_chans,
            args.encoder_embed_dim,
            contain_mask_token=True,
            prepend_cls_token=True,
        )
        # being consistent with Fairseq, which starts from 2 for position embedding
        embed_positions = MutliwayEmbedding(
            modules=[
                PositionalEmbedding(self.vision_embed.num_position_embeddings() + 2, args.encoder_embed_dim),
                PositionalEmbedding(args.max_source_positions, args.encoder_embed_dim),
            ],
            dim=1,
        )
        self.encoder = Encoder(
            args,
            embed_tokens=None,
            embed_positions=embed_positions,
            output_projection=None,
            is_encoder_decoder=False,
        )
        
    def forward(
        self,
        textual_tokens=None,
        visual_tokens=None,
        text_padding_position=None,
        attn_mask=None,
        vision_masked_position=None,
        incremental_state=None,
        positions=None,
    ):
        assert textual_tokens is not None or visual_tokens is not None

        if textual_tokens is None:
            x = self.vision_embed(visual_tokens, vision_masked_position)
            encoder_padding_mask = None
            multiway_split_position = -1
        elif visual_tokens is None:
            x = self.text_embed(textual_tokens)
            encoder_padding_mask = text_padding_position
            multiway_split_position = 0
        else: #VQA here
            x1 = self.vision_embed(visual_tokens, vision_masked_position)
            multiway_split_position = x1.size(1)
            x2 = self.text_embed(textual_tokens)
            x = torch.cat([x1, x2], dim=1)

            if text_padding_position is not None:
                encoder_padding_mask = torch.cat(
                    [
                        torch.zeros(x1.shape[:-1]).to(x1.device).bool(),
                        text_padding_position,
                    ],
                    dim=1,
                )
            else:
                encoder_padding_mask = None

        encoder_out = self.encoder(
            src_tokens=None,
            encoder_padding_mask=encoder_padding_mask,
            attn_mask=attn_mask,
            token_embeddings=x,
            multiway_split_position=multiway_split_position,
            incremental_state=incremental_state,
            positions=positions,
        )
        encoder_out["multiway_split_position"] = multiway_split_position

        return encoder_out

In [11]:
def trunc_normal_(tensor, mean=0., std=1.):
    __call_trunc_normal_(tensor, mean=mean, std=std, a=-std, b=std)


def _get_base_config(
        img_size=224, patch_size=16, drop_path_rate=0,
        checkpoint_activations=None, mlp_ratio=4, vocab_size=64010, **kwargs
):
    return EncoderConfig(
        img_size=img_size, patch_size=patch_size, vocab_size=vocab_size, multiway=True,
        layernorm_embedding=False, normalize_output=True, no_output_layer=True,
        drop_path_rate=drop_path_rate, encoder_embed_dim=768, encoder_attention_heads=12,
        encoder_ffn_embed_dim=int(768 * mlp_ratio), encoder_layers=12,
        checkpoint_activations=checkpoint_activations,
    )


def _get_large_config(
        img_size=224, patch_size=16, drop_path_rate=0,
        checkpoint_activations=None, mlp_ratio=4, vocab_size=64010, **kwargs
):
    return EncoderConfig(
        img_size=img_size, patch_size=patch_size, vocab_size=vocab_size, multiway=True,
        layernorm_embedding=False, normalize_output=True, no_output_layer=True,
        drop_path_rate=drop_path_rate, encoder_embed_dim=1024, encoder_attention_heads=16,
        encoder_ffn_embed_dim=int(1024 * mlp_ratio), encoder_layers=24,
        checkpoint_activations=checkpoint_activations,
    )


class BEiT3Wrapper(nn.Module):
    def __init__(self, args, **kwargs):
        super().__init__()
        self.args = args
        self.beit3 = BEiT3(args)
        self.apply(self._init_weights)

    def fix_init_weight(self):
        def rescale(param, layer_id):
            param.div_(math.sqrt(2.0 * layer_id))

        for layer_id, layer in enumerate(self.blocks):
            rescale(layer.attn.proj.weight.data, layer_id + 1)
            rescale(layer.mlp.fc2.weight.data, layer_id + 1)

    def get_num_layers(self):
        return self.beit3.encoder.num_layers

    @torch.jit.ignore
    def no_weight_decay(self):
        return {'pos_embed', 'cls_token', 'beit3.encoder.embed_positions.A.weight', 'beit3.vision_embed.cls_token', 'logit_scale'}

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

## VQAv2 fine-tuning model

In [12]:
class Pooler(nn.Module):
    def __init__(self, input_features, output_features, norm_layer):
        super().__init__()
        self.norm = norm_layer(input_features)
        self.dense = nn.Linear(input_features, output_features)
        self.activation = nn.Tanh()

    def forward(self, x):
        cls_rep = x[:, 0, :]
        cls_rep = self.norm(cls_rep)
        pooled_output = self.dense(cls_rep)
        pooled_output = self.activation(pooled_output)
        return pooled_output
    
class BEiT3ForVisualQuestionAnswering(BEiT3Wrapper):
    def __init__(
            self,
            args,
            num_classes,
            norm_layer=nn.LayerNorm,
            **kwargs
    ):
        super(BEiT3ForVisualQuestionAnswering, self).__init__(args=args)
        embed_dim = args.encoder_embed_dim
        self.pooler = Pooler(
            input_features=embed_dim,
            output_features=embed_dim,
            norm_layer=norm_layer,
        )
        self.pooler.apply(self._init_weights)
        self.head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 2),
            norm_layer(embed_dim * 2),
            nn.GELU(),
            nn.Linear(embed_dim * 2, num_classes),
        )
        self.head.apply(self._init_weights)

    def forward(self, image, question, padding_mask, **kwargs):
        outputs = self.beit3(
            textual_tokens=question,
            visual_tokens=image,
            text_padding_position=padding_mask,
        )
        x = outputs["encoder_out"]
        cls_rep = self.pooler(x)
        return self.head(cls_rep)

@register_model
def beit3_base_patch16_384_vqav2(pretrained=False, **kwargs):
    args = _get_base_config(img_size=384, **kwargs)
    args.normalize_output = False
    model = BEiT3ForVisualQuestionAnswering(args, num_classes=3129, **kwargs)
    return model

@register_model
def beit3_base_patch16_480_vqav2(pretrained=False, **kwargs):
    args = _get_base_config(img_size=480, **kwargs)
    args.normalize_output = False
    model = BEiT3ForVisualQuestionAnswering(args, num_classes=3129, **kwargs)
    return model

## Load model

In [13]:
import torch
import timm

model_name = "beit3_base_patch16_480_vqav2"

model = timm.models.create_model(model_name, pretrained=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load(model_ckpt_path, map_location=device, weights_only=False)

model.load_state_dict(checkpoint['model'])

<All keys matched successfully>

In [15]:
from transformers import XLMRobertaTokenizer
tokenizer = XLMRobertaTokenizer("/kaggle/input/beit3-main/beit3.spm")

## Create data loader

In [20]:
import os

def create_id_to_filepath(df, assets_path):
    extensions = ['.jpg', '.png', '.jpeg', '.JPG', '.PNG', '.JPEG']
    
    # Tạo dictionary ID to file path
    id_to_filepath = {}
    for img_id in df['Image'].unique():
        for ext in extensions:
            file_path = os.path.join(assets_path, img_id + ext)
            if os.path.exists(file_path):
                id_to_filepath[img_id] = file_path
                break
    
    return id_to_filepath

In [21]:
id2filename_train = create_id_to_filepath(train_df, assets_path)
id2filename_val = create_id_to_filepath(val_df, assets_path)
id2filename_test = create_id_to_filepath(test_df, assets_path)
id2filename = id2filename_train | id2filename_val | id2filename_test

In [22]:
all_answers = pd.concat([train_df['Answer'], val_df['Answer'], test_df['Answer']]).unique()
label2id = {answer: idx for idx, answer in enumerate(all_answers)}
id2label = {idx: answer for answer, idx in label2id.items()}

In [23]:
from torchvision.datasets.folder import default_loader
from torch.nn.utils.rnn import pad_sequence

def to_image_tokens_batch(image_path, input_size=384):
    transform = build_transform(input_size)
    image = default_loader(image_path)
    image = transform(image)
    return image

def to_text_tokens_batch(text, tokenizer, max_len=64):
    tokens_orig = tokenizer.tokenize(text)
    token_ids = tokenizer.convert_tokens_to_ids(tokens_orig)
    
    if len(token_ids) > max_len - 2:
        token_ids = token_ids[:max_len - 2]
    
    tokens = [tokenizer.bos_token_id] + token_ids + [tokenizer.eos_token_id]
    num_tokens = len(tokens)
    padding_mask = [0] * num_tokens + [1] * (max_len - num_tokens)
    tokens_padded = tokens + [tokenizer.pad_token_id] * (max_len - num_tokens)

    padding_mask_tensor = torch.tensor(padding_mask).to(device)
    token_ids_tensor = torch.tensor(tokens_padded).to(device)

    return token_ids_tensor, padding_mask_tensor

def to_text_tokens_list(questions, tokenizer, max_len=64):
    token_ids_tensors = []
    padding_mask_tensors = []
    
    for question in questions:
        token_ids, padding_mask = to_text_tokens_batch(question, tokenizer, max_len)
        token_ids_tensors.append(token_ids)
        padding_mask_tensors.append(padding_mask)
    
    token_ids_tensors = pad_sequence(token_ids_tensors, batch_first=True, padding_value=tokenizer.pad_token_id).to(device)
    padding_mask_tensors = pad_sequence(padding_mask_tensors, batch_first=True, padding_value=1).to(device)
    
    return token_ids_tensors, padding_mask_tensors

In [24]:
from torch.utils.data import Dataset, DataLoader

class VQADataset(Dataset):
    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        question = self.df.iloc[idx]["Question"]
        img_id = self.df.iloc[idx]["Image"]
        
        image = to_image_tokens_batch(id2filename[img_id], input_size=480).to(device) 
        # format labels to tensor
        answer = label2id[self.df.iloc[idx]["Answer"]]
        
        return (image, question, answer)

In [25]:
def collate_fn(batch):
    images, questions, answers = zip(*batch)   
    images = torch.stack(images)
    answers = torch.tensor(answers)
    
    return {
        'pixel_values': images,
        'questions': list(questions),
        'answers': answers
    }

In [27]:
from torch.utils.data import DataLoader

train_dataset = VQADataset(train_df)
train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)

In [28]:
val_dataset = VQADataset(val_df)
val_dataloader = DataLoader(val_dataset, batch_size=8, collate_fn=collate_fn)

## Modify model

In [29]:
model.head[-1] = torch.nn.Linear(in_features=1536, out_features=len(id2label), bias=True)
model.to(device)

BEiT3ForVisualQuestionAnswering(
  (beit3): BEiT3(
    (text_embed): TextEmbedding(64010, 768)
    (vision_embed): VisionEmbedding(
      (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (encoder): Encoder(
      (dropout_module): Dropout(p=0.0, inplace=False)
      (embed_positions): MutliwayEmbedding(
        (A): PositionalEmbedding(903, 768)
        (B): PositionalEmbedding(1024, 768)
      )
      (layers): ModuleList(
        (0-11): 12 x EncoderLayer(
          (self_attn): MultiheadAttention(
            (k_proj): MultiwayNetwork(
              (A): Linear(in_features=768, out_features=768, bias=True)
              (B): Linear(in_features=768, out_features=768, bias=True)
            )
            (v_proj): MultiwayNetwork(
              (A): Linear(in_features=768, out_features=768, bias=True)
              (B): Linear(in_features=768, out_features=768, bias=True)
            )
            (q_proj): MultiwayNetwork(
              (A): Linear(in_features=

In [30]:
# freeze main layer
for param in model.parameters():
    param.requires_grad = False
for param in model.head.parameters():
    param.requires_grad = True
for param in model.beit3.text_embed.parameters():
    param.requires_grad = True

# Training loop

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr = 2e-5, eps = 1e-8)
criterion = nn.CrossEntropyLoss()

EPOCHS = 1000
early_stop_patience = 3
best_val_loss = 1000
epochs_without_improvement = 0
max_grad_norm = 1.0 

for epoch in range(EPOCHS):
    print(f"Epoch: {epoch}")
    model.train()
    train_losses = []
    for batch in tqdm(train_dataloader):
        images = batch['pixel_values'].to(device)
        questions = batch['questions']
        answers = batch['answers'].to(device)
        
        question_ids, padding_mask = to_text_tokens_list(questions, tokenizer, max_len=64)
        
        optimizer.zero_grad()
        outputs = model(image=images, question=question_ids, padding_mask=padding_mask)
        loss = criterion(outputs, answers)
        train_losses.append(loss.item())
        loss.backward()
        
        optimizer.step()
        
    avg_train_loss = np.mean(train_losses)
    
    model.eval()
    val_losses = []
    for val_batch in val_dataloader:
        val_images = val_batch['pixel_values'].to(device)
        val_questions = val_batch['questions']
        val_answers = val_batch['answers'].to(device)
        
        val_question_ids, val_padding_mask = to_text_tokens_list(val_questions, tokenizer, max_len=64)

        with torch.no_grad():
            val_outputs = model(image=val_images, question=val_question_ids, padding_mask=val_padding_mask)
            val_loss = criterion(val_outputs, val_answers)
            val_losses.append(val_loss.item())
    avg_val_loss = np.mean(val_losses)

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_without_improvement = 0 
        torch.save(model.state_dict(), output_path + "/best_model.pth")
    else:
        epochs_without_improvement += 1

    print(f"Loss: {avg_train_loss}, validation loss: {avg_val_loss}")

    if epochs_without_improvement >= early_stop_patience:
        print("Early stopping triggered!")
        break

    torch.cuda.empty_cache()

In [39]:
%cd '/kaggle/working/'

/kaggle/working


In [40]:
from IPython.display import FileLink
FileLink(r'best_model.pth')

/kaggle/working/best_model.pth

# Inference

In [ ]:
model.load_state_dict(torch.load('/kaggle/working/best_model.pth', weights_only=True))

In [ ]:
import torch
from tqdm import tqdm 
from sklearn.metrics import precision_score, recall_score, f1_score

def evaluate_model(model, df, image_col, question_col, answer_col, device, config, id2filename):
    model.eval()
    correct_predictions = 0
    total_predictions = 0
    all_true_answers = []
    all_predicted_answers = []

    with torch.no_grad(): 
        for index, row in tqdm(df.iterrows(), total=len(df), desc="Evaluating"):
            image_path = id2filename[str(row[image_col])]
            question = row[question_col]
            true_answer = row[answer_col] 

            image = to_image_tokens(image_path, input_size=480).to(device)  
            question_ids, padding_mask = to_text_tokens(question, tokenizer, max_len=64)

            output = model(image=image, question=question_ids, padding_mask=padding_mask)
            predicted_class_id = torch.argmax(output, dim=1).item()

            predicted_answer = config.get(predicted_class_id, None) 

            all_true_answers.append(true_answer)
            all_predicted_answers.append(predicted_answer)
            
            if predicted_answer is not None and predicted_answer == true_answer: 
                correct_predictions += 1
            total_predictions += 1

    accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0.0
    precision = precision_score(all_true_answers, all_predicted_answers, average='weighted', zero_division=1)
    recall = recall_score(all_true_answers, all_predicted_answers, average='weighted', zero_division=1)
    f1 = f1_score(all_true_answers, all_predicted_answers, average='weighted', zero_division=1)

    return accuracy, precision, recall, f1, all_true_answers, all_predicted_answers

In [42]:
accuracy, precision, recall, f1, all_true_answers, all_predicted_answers = evaluate_model(model, test_df, image_col="Image", question_col="Question", answer_col="Answer", device=device, config=id2label, id2filename=id2filename_test)
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

Evaluating: 100%|██████████| 3699/3699 [07:09<00:00,  8.60it/s]

Accuracy: 0.4790
Precision: 0.6803
Recall: 0.4790
F1-Score: 0.4116


In [ ]:
import pandas as pd

results_df = pd.DataFrame({
    "Answer": all_true_answers,
    "Predict": all_predicted_answers
})
results_df

## Save result

In [ ]:
results_df.to_csv('/kaggle/working/beit3_results.csv',index=False)